# 01 · Extract

Phase 1: pull the events and fields we need from BigQuery into local Parquet.

Every query lives in `sql/` and is loaded from there, so the notebook and the SQL never drift apart.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from google.cloud import bigquery

# Override with your own BigQuery sandbox project: export GCP_PROJECT=your-project-id
PROJECT_ID = os.environ.get("GCP_PROJECT", "ga4-analytics-project-513720")

# Re-download extracts that already exist in data/raw/
REFRESH = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SQL_DIR = ROOT / "sql"
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

sys.path.insert(0, str(ROOT))
from src.quality import (DROP_CHECKS, PLACEHOLDER_COLS, PLACEHOLDERS, apply_decisions,
                         quality_flags, quality_log)
from src.sessionize import build_sessions, order_events, tie_stats

client = bigquery.Client(project=PROJECT_ID)

In [2]:
def load_sql(filename: str) -> str:
    """Read a query from sql/ and print its dry-run cost before anything runs."""
    sql = (SQL_DIR / filename).read_text()
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    print(f"{filename}: {dry.total_bytes_processed / 1e6:,.1f} MB to process")
    return sql


def run_sql(filename: str) -> pd.DataFrame:
    """Run a small query from sql/ into a DataFrame."""
    return client.query(load_sql(filename)).to_dataframe()


def extract_to_parquet(filename: str, out_path: Path) -> pq.ParquetFile:
    """Run a large query from sql/ and stream the result straight to Parquet via the Storage API."""
    if out_path.exists() and not REFRESH:
        print(f"using existing {out_path.relative_to(ROOT)} (set REFRESH = True to re-extract)")
        return pq.ParquetFile(out_path)
    table =client.query(load_sql(filename)).to_arrow(create_bqstorage_client=True)
    pq.write_table(table, out_path)
    print(f"wrote {out_path.relative_to(ROOT)}: {table.num_rows:,} rows, "
          f"{out_path.stat().st_size / 1e6:,.1f} MB")
    return pq.ParquetFile(out_path)

## Step 1: event inventory

Count every event type in the window to decide which events to keep for sessionizing.

In [3]:
inventory = run_sql("01_event_inventory.sql")
inventory["share"] = inventory["n"] / inventory["n"].sum()
inventory.to_parquet(RAW_DIR / "event_inventory.parquet", index=False)

print(f"{len(inventory)} event types, {inventory['n'].sum():,} events")
inventory.style.format({"n": "{:,}", "share": "{:.2%}"}).hide(axis="index")

01_event_inventory.sql: 56.0 MB to process


17 event types, 4,295,584 events


event_name,n,share
page_view,"1,350,428",31.44%
user_engagement,"1,058,721",24.65%
scroll,"493,072",11.48%
view_item,"386,068",8.99%
session_start,"354,970",8.26%
first_visit,"257,462",5.99%
view_promotion,"190,104",4.43%
add_to_cart,"58,543",1.36%
begin_checkout,"38,757",0.90%
select_item,"31,007",0.72%


**Decision:** extract all 17 event types. BigQuery bills by columns scanned, not rows returned, so filtering events saves nothing. The automatic events (`session_start`, `first_visit`, `user_engagement`) are kept as feature sources and excluded from the *k*-event count when sessionizing.

`view_item_list` appears only 71 times against 31,007 `select_item` events, an obfuscation gap, so it is not used as a feature.

## Step 2: event extraction

One row per event with session ID, timestamp, page, device, geo, first-touch medium and revenue.

In [4]:
events_file = extract_to_parquet("02_extract_events.sql", RAW_DIR / "events.parquet")
events = pd.read_parquet(RAW_DIR / "events.parquet")
events.dtypes

using existing data/raw/events.parquet (set REFRESH = True to re-extract)


session_id                  str
user_pseudo_id              str
event_date                  str
event_timestamp           int64
event_name                  str
session_number            int64
engagement_time_msec    float64
page_location               str
page_title                  str
device                      str
os                          str
country                     str
first_touch_medium          str
revenue                 float64
transaction_id              str
dtype: object

### First look

Quick sanity checks only; the full data-quality checks run after sessionizing.

In [5]:
ts = pd.to_datetime(events["event_timestamp"], unit="us", utc=True)
summary = pd.Series({
    "events": len(events),
    "users": events["user_pseudo_id"].nunique(),
    "sessions (non-null ID)": events["session_id"].nunique(),
    "events with null session_id": events["session_id"].isna().sum(),
    "first event (UTC)": ts.min(),
    "last event (UTC)": ts.max(),
    "distinct event_date": events["event_date"].nunique(),
})
summary.to_frame("value")

,value
events,4295584
users,270154
sessions (non-null ID),360129
events with null session_id,0
first event (UTC),2020-11-01 00:00:04.579566+00:00
last event (UTC),2021-01-31 23:59:55.412363+00:00
distinct event_date,92


In [6]:
# Share of nulls per column, to spot which fields the obfuscation emptied
events.isna().mean().sort_values(ascending=False).to_frame("null share").style.format("{:.2%}")

,null share
revenue,99.87%
transaction_id,82.63%
engagement_time_msec,43.39%
page_title,0.49%
session_id,0.00%
user_pseudo_id,0.00%
event_date,0.00%
event_timestamp,0.00%
event_name,0.00%
session_number,0.00%


## Step 3: item extraction

One row per item attached to an event, unnested from the `items` array. Joins back to the events table on `session_id`, `event_timestamp` and `event_name`.

In [7]:
items_file = extract_to_parquet("03_extract_items.sql", RAW_DIR / "items.parquet")
items = pd.read_parquet(RAW_DIR / "items.parquet")
items.dtypes

using existing data/raw/items.parquet (set REFRESH = True to re-extract)


session_id             str
event_timestamp      int64
event_name             str
item_id                str
item_name              str
item_category          str
price              float64
price_in_usd       float64
quantity           float64
dtype: object

In [8]:
# Which events carry items, and how many per event
per_event = items.groupby(["session_id", "event_timestamp", "event_name"]).size()
(per_event.groupby("event_name").agg(["count", "median", "max"])
 .rename(columns={"count": "events with items", "median": "median items", "max": "max items"})
 .sort_values("events with items", ascending=False))

,events with items,median items,max items
event_name,,,
view_item,241876,12.0,12
view_promotion,134602,1.0,1
add_to_cart,58543,12.0,12
begin_checkout,31991,1.0,31
select_item,31006,12.0,12
select_promotion,8595,1.0,1
purchase,5690,2.0,30
view_item_list,43,12.0,12


In [9]:
# Nulls and placeholders in the item fields
pd.DataFrame({
    "distinct": items.nunique(),
    "null share": items.isna().mean(),
    "placeholder share": items.isin(PLACEHOLDERS).mean(),
}).style.format({"null share": "{:.2%}", "placeholder share": "{:.2%}"})

,distinct,null share,placeholder share
session_id,128474,0.00%,0.00%
event_timestamp,461409,0.00%,0.00%
event_name,8,0.00%,0.00%
item_id,1394,0.00%,3.28%
item_name,431,0.00%,3.73%
item_category,82,0.00%,4.14%
price,72,3.73%,0.00%
price_in_usd,68,99.61%,0.00%
quantity,89,96.23%,0.00%


### Price field

`price_in_usd` is filled only on purchases; `price` covers views, cart and checkout. On purchases, where both exist, they should agree if the store currency is USD.

In [10]:
price_cov = items.groupby("event_name")[["price", "price_in_usd"]].agg(lambda s: s.notna().mean())
display(price_cov.style.format("{:.1%}"))

both = items.dropna(subset=["price", "price_in_usd"])
print(f"rows with both prices: {len(both):,}; price == price_in_usd: "
      f"{(both['price'] - both['price_in_usd']).abs().lt(0.005).mean():.2%}")

,price,price_in_usd
event_name,,
add_to_cart,100.0%,0.0%
begin_checkout,97.5%,0.0%
purchase,97.2%,97.2%
select_item,99.2%,0.0%
select_promotion,0.0%,0.0%
view_item,100.0%,0.0%
view_item_list,98.4%,0.0%
view_promotion,0.0%,0.0%


rows with both prices: 15,555; price == price_in_usd: 100.00%


### What `view_item` actually records

Most `view_item` events carry a full category listing rather than a single product.

In [11]:
view = items[items["event_name"] == "view_item"]
view_events = view.groupby(["session_id", "event_timestamp"]).agg(
    n_items=("item_id", "size"), n_categories=("item_category", "nunique"))
print(f"view_item events with items: {len(view_events):,}")
print(f"  exactly 1 item:   {(view_events['n_items'] == 1).mean():.1%}")
print(f"  exactly 12 items: {(view_events['n_items'] == 12).mean():.1%}")
print(f"  12-item events within one category: "
      f"{(view_events.loc[view_events['n_items'] == 12, 'n_categories'] == 1).mean():.1%}")

view_item events with items: 241,876
  exactly 1 item:   3.6%
  exactly 12 items: 90.8%
  12-item events within one category: 47.9%


## Step 4: sessionize

Order events within each session by timestamp. When events share a timestamp, break the tie by funnel stage (`EVENT_ORDER` in `src/sessionize.py`), so a simultaneous cart and purchase never come out reversed.

`action_index` counts shopper actions only (automatic events excluded); Phase 3 cuts the *k*-event prefixes on it.

In [12]:
ordered = order_events(events)
tie_stats(ordered).to_frame("share").style.format("{:.2%}")

,share
events in a timestamp tie,57.04%
sessions with any tie,99.65%
shopper actions tied with another shopper action,42.89%


In [13]:
# Which event pairs tie most often (first two events of each tied group)
tied = ordered[ordered["ts_tie"]]
tie_groups = tied.groupby(["session_id", "event_timestamp"])["event_name"].agg(" + ".join)
tie_groups.value_counts().head(10).to_frame("tied groups")

,tied groups
event_name,
session_start + first_visit + page_view,257409
page_view + view_item,140482
page_view + scroll,129979
page_view + view_promotion,100600
session_start + page_view,61903
page_view + user_engagement,59891
scroll + user_engagement,57295
session_start + user_engagement,31811
view_item + user_engagement,25419


### One row per session

In [14]:
sessions = build_sessions(ordered)
print(f"sessions: {len(sessions):,}")
print(f"converted: {sessions['converted'].sum():,} ({sessions['converted'].mean():.2%})")
print(f"context changes mid-session: {sessions['context_varies'].mean():.2%}")
sessions.head()

sessions: 360,129
converted: 4,848 (1.35%)
context changes mid-session: 0.00%


,session_id,start_ts,end_ts,n_events,n_actions,engagement_time_msec,revenue,user_pseudo_id,session_number,device,...,n_user_engagement,n_add_to_cart,n_begin_checkout,n_add_shipping_info,n_add_payment_info,n_purchase,start_time,end_time,duration_s,converted
0,10001363.4360935308-5983736405,1607746277508078,1607746292809866,4,1,15285.0,0.0,10001363.4360935308,1,desktop,...,1,0,0,0,0,0,2020-12-12 04:11:17.508078+00:00,2020-12-12 04:11:32.809866+00:00,15.301788,0
1,1000223163.8035209215-6063162078,1610044534089387,1610044544561794,6,3,6757.0,0.0,1000223163.8035209215,1,mobile,...,1,0,0,0,0,0,2021-01-07 18:35:34.089387+00:00,2021-01-07 18:35:44.561794+00:00,10.472407,0
2,1000299.7413851356-3338398581,1611140699887247,1611140705779925,5,2,5811.0,0.0,1000299.7413851356,1,desktop,...,1,0,0,0,0,0,2021-01-20 11:04:59.887247+00:00,2021-01-20 11:05:05.779925+00:00,5.892678,0
3,1000300.3223254235-3614622791,1604497410570448,1604497410570448,2,0,0.0,0.0,1000300.3223254235,2,desktop,...,1,0,0,0,0,0,2020-11-04 13:43:30.570448+00:00,2020-11-04 13:43:30.570448+00:00,0.000000,0
4,1000300.3223254235-9350310735,1604481671657014,1604481678760672,4,2,1850.0,0.0,1000300.3223254235,1,desktop,...,0,0,0,0,0,0,2020-11-04 09:21:11.657014+00:00,2020-11-04 09:21:18.760672+00:00,7.103658,0


In [15]:
sessions[["n_events", "n_actions", "duration_s", "engagement_time_msec"]].describe(
    percentiles=[0.5, 0.9, 0.99]).T.style.format("{:,.1f}")

,count,mean,std,min,50%,90%,99%,max
n_events,"360,129.0",11.9,26.5,1.0,5.0,22.0,125.0,"1,007.0"
n_actions,"360,129.0",7.3,18.3,0.0,3.0,15.0,90.0,737.0
duration_s,"360,129.0",203.3,936.3,0.0,14.0,610.9,"2,456.5","402,477.1"
engagement_time_msec,"360,129.0","70,345.0","293,438.0",0.0,"8,245.0","162,616.0","1,019,304.8","70,799,011.0"


In [16]:
# Converted vs not: medians of the basic session profile
sessions.groupby("converted")[["n_events", "n_actions", "duration_s", "n_view_item", "n_add_to_cart"]].median()

,n_events,n_actions,duration_s,n_view_item,n_add_to_cart
converted,,,,,
0,5.0,2.0,13.283020,0.0,0.0
1,84.0,61.0,1078.194847,9.0,1.0


## Step 5: data-quality checks

One boolean flag per session per check (`src/quality.py`). The log shows how many sessions each check touches, overall and among converters, since checks that hit converters move the label.

In [17]:
flags = quality_flags(sessions, ordered)
dq_log = quality_log(flags, sessions.set_index("session_id")["converted"])
dq_log.style.format({
    "sessions affected": "{:,}", "converted sessions affected": "{:,}",
    "share of all sessions": "{:.2%}", "share of converted sessions": "{:.2%}",
    "conversion rate in group": "{:.2%}",
})

,sessions affected,share of all sessions,converted sessions affected,share of converted sessions,conversion rate in group,decision
check,,,,,,
missing_session_id,0,0.00%,0,0.00%,nan%,drop (none found)
no_session_start,"5,272",1.46%,103,2.12%,1.95%,"keep, flag"
window_start_edge,3,0.00%,0,0.00%,0.00%,drop
window_end_edge,77,0.02%,1,0.02%,1.30%,drop
purchase_without_browsing,175,0.05%,175,3.61%,100.00%,drop
purchase_without_cart,"2,031",0.56%,"2,031",41.89%,100.00%,"keep, flag (Task B scope)"
bad_revenue,411,0.11%,411,8.48%,100.00%,"keep label, flag revenue"
bad_transaction_id,813,0.23%,813,16.77%,100.00%,"keep, flag"
duplicate_purchases,552,0.15%,552,11.39%,100.00%,"keep label, dedupe revenue"


### Detail: duplicate purchases and transaction IDs

In [18]:
purchases = ordered[ordered["event_name"] == "purchase"]
per_session = purchases.groupby("session_id").agg(n=("event_name", "size"), tids=("transaction_id", "nunique"))
multi = per_session[per_session["n"] > 1]
print(f"sessions with >1 purchase: {len(multi):,}")
print(f"  a transaction ID repeats: {(multi['tids'] < multi['n']).sum():,}")
print(f"  all transaction IDs distinct: {(multi['tids'] == multi['n']).sum():,}")
print(f"repeated (session, transaction_id, revenue) rows: "
      f"{purchases.duplicated(['session_id', 'transaction_id', 'revenue']).sum():,}")

valid = purchases[~purchases["transaction_id"].isin(PLACEHOLDERS) & purchases["transaction_id"].notna()]
print(f"valid transaction IDs seen in more than one session: "
      f"{(valid.groupby('transaction_id')['session_id'].nunique() > 1).sum()} of {valid['transaction_id'].nunique():,}")

sessions with >1 purchase: 552
  a transaction ID repeats: 330
  all transaction IDs distinct: 222
repeated (session, transaction_id, revenue) rows: 394
valid transaction IDs seen in more than one session: 15 of 4,451


### Detail: placeholder values in context fields

In [19]:
pd.DataFrame({
    col: sessions[col].isin(PLACEHOLDERS).mean() for col in PLACEHOLDER_COLS
}, index=["placeholder share of sessions"]).T.style.format("{:.2%}")

,placeholder share of sessions
device,0.00%
os,2.52%
country,0.80%
first_touch_medium,20.74%


## Step 6: apply decisions and save

Decisions live in `DECISIONS` in `src/quality.py`. The log is saved to `reports/data_quality_log.csv` and the clean tables to `data/processed/`.

In [20]:
dq_log.to_csv(ROOT / "reports" / "data_quality_log.csv", float_format="%.6f")

sessions_clean, events_clean, items_clean = apply_decisions(sessions, ordered, items, flags)

dropped = len(sessions) - len(sessions_clean)
print(f"dropped {dropped:,} sessions ({dropped / len(sessions):.2%}) on: {', '.join(DROP_CHECKS)}")
print(f"sessions:  {len(sessions_clean):,}")
print(f"converted: {sessions_clean['converted'].sum():,}")
print(f"conversion rate: {sessions_clean['converted'].mean():.2%}")
print(f"duplicate purchase rows marked: {events_clean['is_duplicate_purchase'].sum():,}")
print(f"items without a matching event: {items_clean['event_index'].isna().sum():,}")

dropped 281 sessions (0.08%) on: missing_session_id, window_start_edge, window_end_edge, purchase_without_browsing, duration_over_4h
sessions:  359,848
converted: 4,672
conversion rate: 1.30%
duplicate purchase rows marked: 367
items without a matching event: 0


In [21]:
for name, df in [("sessions", sessions_clean), ("events_clean", events_clean), ("items_clean", items_clean)]:
    path = PROCESSED_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"wrote {path.relative_to(ROOT)}: {len(df):,} rows, {path.stat().st_size / 1e6:,.1f} MB")

wrote data/processed/sessions.parquet: 359,848 rows, 32.3 MB


wrote data/processed/events_clean.parquet: 4,283,472 rows, 68.6 MB


wrote data/processed/items_clean.parquet: 3,974,878 rows, 42.1 MB
